# Lab 16 — Multi-agent evaluation harness from scratch

Build the evaluation harness for Path 03's multi-agent systems. Five trajectory metrics + two outcome metrics, applied to a hand-curated `trace_set.jsonl` of 15 recorded runs from Labs 10/11/12.

The pattern extends [Lab 09's RAG eval harness](../09-evaluating-agentic-rag/) for trajectories. Same shape: hand-curated fixtures, rule-based tier, optional LLM-as-judge tier, category slicing. Different unit of analysis — a multi-agent trajectory instead of a single query/answer pair.

> ⏱ Run time: 100-130 min including reading.
> 📖 Read [`concepts/multi-agent/multi-agent-evaluation.md`](../../concepts/multi-agent/multi-agent-evaluation.md) and [`concepts/multi-agent/trajectory-level-metrics.md`](../../concepts/multi-agent/trajectory-level-metrics.md) first.
> ⬅️ The trace fixtures replay Lab 10/11/12 trajectories. Familiarity with those patterns is required to read the traces.


## Step 0: Setup

No new dependencies. The optional Step 9 (LLM-as-judge) uses `openai` or `anthropic`; everything else is pure Python + `pydantic` + `pandas`.

In [ ]:
import json
import os
import pathlib
import re
from collections import Counter
from typing import Literal

import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

# Provider config — only needed if Step 9 (LLM-as-judge) is run
PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]

print(f"Python {os.sys.version.split()[0]}, pandas {pd.__version__}")


## Step 1: Load and validate the trace set

The trace set is `trace_set.jsonl` next to this notebook. Each line is one recorded multi-agent run. We validate each entry with strict pydantic models — `extra="forbid"` so typos in fixtures fail loudly rather than passing silently.

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class TraceStep(StrictModel):
    """One step in a recorded trajectory.

    `node` is the agent that executed (supervisor, researcher, writer, critic,
    planner, executor, synthesizer). `args` is what it received; `output` is
    what it returned. `status` is "ok" by default; "error", "step_cap", or
    "wrong_tool" for failure modes.
    """
    node: str
    args: dict
    output: dict
    status: str = "ok"


class Trace(StrictModel):
    """One recorded multi-agent run, end-to-end."""
    id: str
    source_lab: Literal["lab10", "lab11", "lab12"]
    task: str
    trajectory: list[TraceStep]
    final_answer: str
    expected_handoffs: list[str]      # golden routing sequence
    expected_citations: list[str]     # golden citation URLs
    category: Literal["happy_path", "tool_failure", "replan_needed",
                      "citation_drift", "step_cap_hit"]


# Load and validate
TRACE_SET_PATH = pathlib.Path("./trace_set.jsonl")
assert TRACE_SET_PATH.exists(), (
    f"Cannot find {TRACE_SET_PATH}. "
    "This notebook expects to be run from its containing directory."
)

traces: list[Trace] = []
errors = []
with TRACE_SET_PATH.open() as f:
    for line_num, line in enumerate(f, 1):
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
            traces.append(Trace.model_validate(obj))
        except (json.JSONDecodeError, ValidationError) as e:
            errors.append((line_num, e))

if errors:
    print(f"⚠ {len(errors)} fixture validation errors:")
    for ln, e in errors[:5]:
        print(f"  line {ln}: {e}")
else:
    print(f"✓ All {len(traces)} traces validated")

# Distribution check
print()
print(f"Categories: {dict(Counter(t.category for t in traces))}")
print(f"Source labs: {dict(Counter(t.source_lab for t in traces))}")


**Sample output**:

```
✓ All 15 traces validated

Categories: {'happy_path': 5, 'citation_drift': 3, 'tool_failure': 2, 'step_cap_hit': 3, 'replan_needed': 2}
Source labs: {'lab10': 5, 'lab11': 5, 'lab12': 5}
```

The trace set is balanced 5/5/5 across source labs and spans all five categories. Some categories (`citation_drift`, `step_cap_hit`) have 3 traces each; others (`tool_failure`, `replan_needed`) have 2. The category coverage is what gives the harness diagnostic value — see Step 7 for how the slice unwraps an aggregate.

## Step 2: Trajectory metrics — handoff success rate, routing accuracy

The first two trajectory metrics. Both look at the supervisor's routing decisions and the structural integrity of inter-agent handoffs.

**Handoff success rate**: fraction of inter-agent handoffs where the payload was well-formed and the upstream agent succeeded. A handoff from researcher to writer is "successful" when (a) the researcher returned `status="ok"` and (b) the payload contained the expected fields. The metric measures the *rate* of successful handoffs across the trajectory.

**Routing accuracy**: fraction of routing decisions that matched `expected_handoffs`. We compare the actual sequence of nodes visited against the golden sequence. Use sequence alignment (longest common subsequence) rather than strict equality — a trace with one extra supervisor visit (because the LLM looped briefly) shouldn't get 0.0 just for the deviation.

In [ ]:
def handoff_success_rate(trace: Trace) -> float:
    """Fraction of inter-agent handoffs that delivered a well-formed payload
    with status="ok" from the source agent.

    A handoff is each step except the first. For each handoff, check:
    - the previous step's status was "ok" (or a recoverable status like "step_cap"
      that was correctly surfaced — we count step_cap as a successful handoff
      because the downstream agent should handle it)
    - the current step received non-empty args
    """
    if len(trace.trajectory) < 2:
        return 1.0  # No handoffs; trivially successful

    successful = 0
    total_handoffs = 0
    for i in range(1, len(trace.trajectory)):
        prev = trace.trajectory[i - 1]
        curr = trace.trajectory[i]
        total_handoffs += 1
        # Successful if upstream completed cleanly AND downstream received non-empty args
        # "step_cap" counts as successful — it's a contracted failure mode the next
        # agent is expected to handle
        upstream_ok = prev.status in ("ok", "step_cap")
        downstream_received = bool(curr.args)
        if upstream_ok and downstream_received:
            successful += 1

    return successful / total_handoffs if total_handoffs > 0 else 1.0


def routing_accuracy(trace: Trace) -> float:
    """Fraction of nodes in the actual trajectory that match expected_handoffs
    via longest common subsequence.

    LCS-based so we don't penalize traces with extra steps (e.g., an extra
    supervisor visit) as heavily as traces with skipped agents.
    """
    actual = [s.node for s in trace.trajectory]
    expected = trace.expected_handoffs

    # LCS dynamic programming
    n, m = len(actual), len(expected)
    if n == 0 or m == 0:
        return 0.0

    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n):
        for j in range(m):
            if actual[i] == expected[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    lcs_length = dp[n][m]
    # Normalize by expected length — penalizes missing nodes harder than extra ones
    return lcs_length / m


# Run on the trace set
print(f"{'trace_id':<15} {'category':<18} {'handoff_succ':>12} {'routing_acc':>12}")
print("─" * 60)
for t in traces:
    h = handoff_success_rate(t)
    r = routing_accuracy(t)
    print(f"{t.id:<15} {t.category:<18} {h:>12.2f} {r:>12.2f}")


**Sample output** (a few interesting rows):

```
trace_id        category           handoff_succ  routing_acc
─────────────────────────────────────────────────────────────
lab10_t01       happy_path                 1.00         1.00
lab10_t05       happy_path                 1.00         0.50   ← routing bug: skipped researcher
lab10_t03       tool_failure               0.75         1.00   ← upstream tool error
lab12_t05       step_cap_hit               0.33         1.00   ← planner retries all error-statused
```

A few patterns to notice:

- `lab10_t05` is annotated as `happy_path` (the *user task* is happy) but has routing_accuracy 0.50. That's the diagnostic signal — the category says "should work end-to-end," routing accuracy says "it didn't." Category and metrics together localize the bug to routing.
- `lab10_t03` has handoff_success_rate 0.75 (the failed `web_search` step) but routing_accuracy 1.00 (routing was correct; the tool was the problem). Tool failures show up here, routing bugs show up there.
- `lab12_t05` (planner cap) has handoff_success 0.33: of three handoffs, only one (planner→synthesizer) was clean. The first two were planner→planner retries where the upstream returned status=`error`. Routing is still 1.00 — the planner correctly escalated to the synthesizer after exhausting its retry budget. The two metrics together tell you "structurally degraded but routed correctly" — which is exactly what the trace was designed to test.


## Step 3: Plan-specific metrics — plan validity, plan coverage

These metrics only apply to Lab 12-style trajectories (planner-executor pattern). They return `None` for Lab 10/11 traces.

**Plan validity**: did the planner emit a valid plan that passed `validate_graph()`? We extract the planner's output and check the `plan_valid` flag. Binary — either the plan was valid or it wasn't. For traces with planner retries (the planner emits an invalid plan, retries, emits a valid plan), we score the *final* plan that was actually used.

**Plan coverage**: of the steps in the final plan, what fraction executed cleanly? Coverage = steps with `status="ok"` / total plan steps. A plan can be valid but have low coverage when tools fail mid-execution.

In [ ]:
def plan_validity(trace: Trace) -> float | None:
    """Returns 1.0 if the final plan emitted by the planner was valid,
    0.0 if no valid plan was produced, None for non-Lab-12 traces.
    """
    if trace.source_lab != "lab12":
        return None

    # Find the last planner step and check its plan_valid flag
    planner_steps = [s for s in trace.trajectory if s.node == "planner"]
    if not planner_steps:
        return 0.0
    final_planner = planner_steps[-1]
    is_valid = final_planner.output.get("plan_valid", False)
    return 1.0 if is_valid else 0.0


def plan_coverage(trace: Trace) -> float | None:
    """Fraction of plan steps that executed with status="ok".

    Reads the final plan's step count from the planner's output. Counts
    executor steps with status="ok" against the total. Returns None for
    non-Lab-12 traces.
    """
    if trace.source_lab != "lab12":
        return None

    # Get the final (valid) plan's step count
    planner_steps = [s for s in trace.trajectory if s.node == "planner"]
    if not planner_steps:
        return 0.0
    final_plan = planner_steps[-1].output.get("plan", {})
    plan_step_count = len(final_plan.get("steps", []))
    if plan_step_count == 0:
        return 0.0  # No steps to cover

    # Count executor steps with status="ok"
    successful_executors = sum(
        1 for s in trace.trajectory
        if s.node == "executor" and s.status == "ok"
    )
    # Cap at plan step count — a buggy trace with extra executors shouldn't
    # exceed 1.0
    return min(successful_executors / plan_step_count, 1.0)


# Run on the trace set — only Lab 12 traces return numeric values
print(f"{'trace_id':<15} {'category':<18} {'plan_validity':>14} {'plan_coverage':>14}")
print("─" * 65)
for t in traces:
    v = plan_validity(t)
    c = plan_coverage(t)
    v_str = f"{v:.2f}" if v is not None else "N/A"
    c_str = f"{c:.2f}" if c is not None else "N/A"
    print(f"{t.id:<15} {t.category:<18} {v_str:>14} {c_str:>14}")


**Sample output** (Lab 12 traces only — others show N/A):

```
trace_id        category           plan_validity  plan_coverage
──────────────────────────────────────────────────────────────
lab12_t01       happy_path                  1.00           1.00
lab12_t02       replan_needed               1.00           1.00   ← invalid plan retried; final plan valid
lab12_t03       tool_failure                1.00           0.67   ← 2 of 3 steps succeeded; one HTTP 503
lab12_t04       replan_needed               1.00           1.00   ← unknown_tool caught by validator; retried successfully
lab12_t05       step_cap_hit                0.00           0.00   ← planner exhausted retries without a valid plan
```

`plan_validity` measures "did the planner produce something the validator accepted." `plan_coverage` measures "did execution succeed." They split the planner-side and executor-side reliability concerns. A `validity=1.0, coverage=0.5` reading says the plan was correct but execution went wrong (tool failures, environmental issues). A `validity=0.0` reading says the planner failed; coverage is moot.

## Step 4: Set-level metric — replan rate

Replan rate is the only metric that operates on the whole trace set, not a single trace. It counts what fraction of traces involved at least one planner retry or replan.

In Lab 12's pattern, a "replan" is any planner invocation beyond the first. Two replan flavors:

- **Validation-retry replan**: the planner emitted an invalid plan; the validator caught it; the planner retried within its retry budget. This is a tight loop inside the planner.
- **Execution-failure replan**: the executor failed mid-plan; the supervisor escalated back to the planner for a new plan. This is a longer loop through the synthesizer's replan signal.

For Lab 16's harness we count both as "replan needed." The implementation just counts how many planner steps appear in each trace — anything more than 1 means a retry/replan happened.

In [ ]:
def replan_rate(trace_set: list[Trace]) -> dict:
    """Set-level metric: fraction of (Lab 12) traces that had >1 planner step.

    Returns a dict with the rate plus diagnostic counts.
    """
    lab12_traces = [t for t in trace_set if t.source_lab == "lab12"]
    if not lab12_traces:
        return {"rate": 0.0, "with_replans": 0, "total_lab12": 0,
                "mean_replans_per_trace": 0.0}

    with_replans = 0
    total_replans = 0
    for t in lab12_traces:
        planner_step_count = sum(1 for s in t.trajectory if s.node == "planner")
        # Each planner step beyond the first is a replan
        replans = max(planner_step_count - 1, 0)
        total_replans += replans
        if replans > 0:
            with_replans += 1

    return {
        "rate": with_replans / len(lab12_traces),
        "with_replans": with_replans,
        "total_lab12": len(lab12_traces),
        "mean_replans_per_trace": total_replans / len(lab12_traces),
    }


rr = replan_rate(traces)
print("Replan-rate summary (Lab 12 traces only):")
print(f"  Traces with ≥1 replan: {rr['with_replans']} / {rr['total_lab12']}")
print(f"  Replan rate:           {rr['rate']:.2f}")
print(f"  Mean replans per run:  {rr['mean_replans_per_trace']:.2f}")


**Sample output**:

```
Replan-rate summary (Lab 12 traces only):
  Traces with ≥1 replan: 3 / 5
  Replan rate:           0.60
  Mean replans per run:  0.80
```

3 of 5 Lab 12 traces replanned at least once. That sounds high but the trace set is *deliberately* skewed — it's a diagnostic fixture, not a production sample. Two `replan_needed` traces by design + one `step_cap_hit` that maxed out the planner retries = three traces with replan activity. The lab's lesson: trace-set composition determines what the aggregate metric means. Don't compare set-level metrics across trace sets with different category distributions.

In a production replay set with 80% happy-path traces, you'd expect replan rate around 5-15%. Above 20% in production signals the planner is brittle or the tool registry is incomplete.

## Step 5: Outcome metric — citation preservation across handoffs

The first outcome metric. Lab 10/11's researcher emits a list of citations; the writer is supposed to preserve them. Citation preservation = `|expected ∩ final| / |expected|`.

The implementation has one subtle piece: URL canonicalization. The same web page can be cited as `https://example.com/x`, `https://example.com/x/`, or `https://example.com/x?utm=foo` — all the same page in practice. Without canonicalization, you get false negatives on what are actually preserved citations.

In [ ]:
_TRACKING_PARAMS = {"utm_source", "utm_medium", "utm_campaign", "utm_term",
                      "utm_content", "ref", "fbclid", "gclid"}


def _canonicalize_url(url: str) -> str:
    """Normalize a URL for citation comparison.

    Strips: trailing slash, fragment, tracking-only query params.
    Preserves: scheme, host, path, content-bearing query params.
    """
    from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

    parsed = urlparse(url.strip())
    # Strip fragment
    fragment = ""
    # Strip trailing slash from path (but keep root "/")
    path = parsed.path.rstrip("/") if parsed.path != "/" else parsed.path
    # Filter tracking params
    qs = parse_qsl(parsed.query, keep_blank_values=True)
    qs_filtered = [(k, v) for k, v in qs if k not in _TRACKING_PARAMS]
    query = urlencode(qs_filtered)
    return urlunparse((parsed.scheme.lower(), parsed.netloc.lower(),
                        path, parsed.params, query, fragment))


def _extract_final_citations(trace: Trace) -> list[str]:
    """Extract citation URLs from the final answer text.

    The final answer typically has citations in the format:
        [N] Title — URL
    We pull the URLs via regex.
    """
    # Match http(s) URLs in the final answer
    url_pattern = re.compile(r'https?://[^\s\]\)\)<>"\s]+')
    raw_urls = url_pattern.findall(trace.final_answer)
    return [_canonicalize_url(u) for u in raw_urls]


def citation_preservation(trace: Trace) -> dict:
    """Returns a dict with three subscores:
    - preservation: fraction of expected citations that appear in final
    - hallucinated_count: count of citations in final not in expected
    - dropped_count: count of expected citations missing from final
    """
    expected = {_canonicalize_url(u) for u in trace.expected_citations}
    final_set = set(_extract_final_citations(trace))

    if not expected:
        # No citations expected — score is 1.0 if final has none, else 0.0
        return {
            "preservation": 1.0 if not final_set else 0.0,
            "hallucinated_count": len(final_set),
            "dropped_count": 0,
        }

    preserved = expected & final_set
    hallucinated = final_set - expected
    dropped = expected - final_set
    return {
        "preservation": len(preserved) / len(expected),
        "hallucinated_count": len(hallucinated),
        "dropped_count": len(dropped),
    }


# Run on the trace set
print(f"{'trace_id':<15} {'category':<18} {'preserv':>8} {'hallu':>6} {'dropped':>8}")
print("─" * 60)
for t in traces:
    cp = citation_preservation(t)
    print(f"{t.id:<15} {t.category:<18} {cp['preservation']:>8.2f} "
          f"{cp['hallucinated_count']:>6} {cp['dropped_count']:>8}")


**Sample output** (interesting rows):

```
trace_id        category           preserv  hallu  dropped
─────────────────────────────────────────────────────────────
lab10_t01       happy_path            1.00       0        0
lab10_t02       citation_drift        0.67       0        1   ← writer dropped one citation
lab10_t05       happy_path            0.00       0        2   ← routing bug; no research ran
lab11_t03       citation_drift        0.50       0        1   ← writer revision dropped citation
lab11_t04       citation_drift        0.50       0        1   ← writer compressed; lost one
lab12_t05       step_cap_hit          1.00       0        0   ← no citations expected; final has none
```

The `citation_drift` category traces all score below 1.0 on preservation — by design, since the category exists to surface this failure mode. The aggregate citation_preservation across the full set hides the picture; the per-category slice in Step 7 will surface it clearly.

`lab10_t05` is a useful subtle case: the routing bug (skipped researcher) cascades into 0.0 preservation because the writer had no citations to preserve. The metric correctly attributes the symptom even though the root cause is in routing — that's what category slicing helps disambiguate.

## Step 6: Outcome metric — groundedness (rule-based)

The second outcome metric. Same shape as Lab 09's groundedness — we ask "is every factual claim in the final answer supported by content the system actually saw?"

The implementation is the rule-based version: lexical overlap between claims and supporting content. It's conservative (paraphrased-but-true claims get marked unsupported) but it's free and deterministic. Lab 09's lesson on the rule-based vs LLM-as-judge trade-off applies.

For multi-agent we collect the "supporting content" from all researcher/executor outputs across the trajectory — anything the writer or synthesizer could have plausibly used as input.

In [ ]:
def _extract_claims(answer: str) -> list[str]:
    """Split the final answer into sentence-level claims.

    Simple sentence splitter — splits on period+space, drops citation
    listings (lines starting with "[N]") since those aren't claims.
    """
    # Strip the citation listing at the end
    body_lines = []
    for line in answer.split("\n"):
        if re.match(r"^\[\d+\]", line.strip()):
            continue
        body_lines.append(line)
    body = " ".join(body_lines).strip()

    # Split on . followed by space or end-of-string
    sentences = re.split(r"\.\s+|\.$", body)
    claims = [s.strip() for s in sentences if len(s.strip()) > 10]
    return claims


def _extract_supporting_content(trace: Trace) -> str:
    """Concatenate all content the writer/synthesizer plausibly saw."""
    parts = []
    for s in trace.trajectory:
        if s.node in ("researcher", "executor", "retriever"):
            # Look for findings, results, text, etc.
            for key in ("findings", "text", "result"):
                if key in s.output:
                    val = s.output[key]
                    parts.append(str(val))
    return " ".join(parts).lower()


def _lexical_overlap(claim: str, content: str, threshold: float = 0.5) -> bool:
    """A claim is grounded if at least `threshold` of its content words appear
    in `content`. Content words = words >=4 chars after stripping punctuation
    and stopwords.
    """
    stopwords = {"the", "and", "for", "with", "from", "have", "this", "that",
                  "these", "those", "their", "into", "been", "will", "also",
                  "more", "than", "such", "some", "most", "other", "which"}
    words = re.findall(r"\w{4,}", claim.lower())
    content_words = [w for w in words if w not in stopwords]
    if not content_words:
        return True  # Trivially grounded — no content words to check
    matches = sum(1 for w in content_words if w in content)
    return (matches / len(content_words)) >= threshold


def groundedness(trace: Trace) -> dict:
    """Rule-based groundedness: lexical-overlap between claims and content
    the system saw.

    Returns {grounded_fraction, total_claims, grounded_count}.
    """
    claims = _extract_claims(trace.final_answer)
    if not claims:
        return {"grounded_fraction": 1.0, "total_claims": 0, "grounded_count": 0}

    content = _extract_supporting_content(trace)
    grounded = sum(1 for c in claims if _lexical_overlap(c, content))
    return {
        "grounded_fraction": grounded / len(claims),
        "total_claims": len(claims),
        "grounded_count": grounded,
    }


# Run on the trace set
print(f"{'trace_id':<15} {'category':<18} {'grounded':>10} {'claims':>8}")
print("─" * 55)
for t in traces:
    g = groundedness(t)
    print(f"{t.id:<15} {t.category:<18} {g['grounded_fraction']:>10.2f} "
          f"{g['total_claims']:>8}")


**Sample output** (interesting rows):

```
trace_id        category             grounded   claims
───────────────────────────────────────────────────────
lab10_t01       happy_path               0.50        2
lab10_t05       happy_path               0.00        2   ← no research ran; claims ungrounded
lab11_t04       citation_drift           0.00        1   ← unsupported claim ("47 of top 50")
lab12_t04       replan_needed            0.00        1   ← brief content; rule too strict
lab12_t05       step_cap_hit             0.00        1   ← refusal-style; rule-based misses the refusal signal
```

The rule-based version is intentionally conservative: it asks "do at least 50% of the claim's content words appear in the supporting content." That works for fact-heavy claims that quote the source. It fails for:

- **Paraphrased-but-correct claims**: `lab10_t01`'s final answer paraphrases the researcher's findings (an "open standard for connecting LLMs" becomes "open standard for connecting LLMs to external tools and data sources"). One claim survives the threshold; one doesn't.
- **Brief supporting content** (`lab12_t04`): when the fetched page text is short, claim-against-content overlap is brittle.
- **Principled refusals** (`lab12_t05`): the answer "Could not produce a valid plan..." has no support to find. The rule-based metric scores this as 0.0 — the metric can't distinguish "ungrounded fabrication" from "ungrounded refusal." LLM-as-judge handles this distinction; rule-based doesn't.

`lab11_t04` is the diagnostic case the rule-based version *correctly* catches: the writer fabricated "47 of the top 50 AI companies" — no content overlap with the researcher's brief. Real failure detected by a 30-line metric.

The aggregate groundedness across the trace set will be low — that's the metric being conservative, not the system being broken. For shipping decisions, pair rule-based with LLM-as-judge or human spot-checks on a sample.


## Step 7: Run the full harness — comparison table and category slicing

Now we run all seven metrics over the full trace set and assemble a comparison table. Then we aggregate, then we slice by category — Lab 09's discipline carried forward.

In [ ]:
def run_harness(trace_set: list[Trace]) -> pd.DataFrame:
    """Run all per-trace metrics over the trace set; return a DataFrame."""
    rows = []
    for t in trace_set:
        cp = citation_preservation(t)
        g = groundedness(t)
        rows.append({
            "trace_id": t.id,
            "source_lab": t.source_lab,
            "category": t.category,
            "handoff_success": handoff_success_rate(t),
            "routing_accuracy": routing_accuracy(t),
            "plan_validity": plan_validity(t),
            "plan_coverage": plan_coverage(t),
            "citation_preservation": cp["preservation"],
            "hallucinated_citations": cp["hallucinated_count"],
            "groundedness": g["grounded_fraction"],
            "claim_count": g["total_claims"],
        })
    return pd.DataFrame(rows)


df = run_harness(traces)
print("Per-trace metrics (full table):")
print(df.to_string(index=False))


In [ ]:
# Aggregate without slicing — the "lying" number
print("\nAggregate (across all traces — misleading):")
print(df[["handoff_success", "routing_accuracy", "citation_preservation",
          "groundedness"]].mean().round(3))

print("\nAggregate sliced by category:")
by_cat = df.groupby("category")[
    ["handoff_success", "routing_accuracy", "citation_preservation", "groundedness"]
].mean().round(3)
print(by_cat)


**Sample output**:

```
Aggregate (across all traces — misleading):
handoff_success          0.892
routing_accuracy         0.967
citation_preservation    0.811
groundedness             0.411

Aggregate sliced by category:
                handoff_success  routing_accuracy  citation_preservation  groundedness
category
citation_drift            1.000             1.000                  0.556         0.222
happy_path                1.000             0.900                  0.800         0.500
replan_needed             0.775             1.000                  1.000         0.500
step_cap_hit              0.778             1.000                  1.000         0.167
tool_failure              0.750             1.000                  0.750         0.750
```

The slice tells the real story behind each aggregate:

- **`citation_preservation = 0.811` aggregate** hides that `citation_drift` traces preserve only 56% (the failure mode they were designed to surface, working as intended) while `happy_path` runs preserve 80% — that 20% drop on healthy paths is a real issue worth tracking.
- **`groundedness = 0.411` aggregate** looks alarming until you see the slice. `tool_failure` traces ground at 0.75 because the recoverable cases preserve claim-to-content overlap. `citation_drift` and `step_cap_hit` ground low because they include principled refusals and brief-content cases the rule-based metric punishes. The aggregate without the slice would read as "the system hallucinates"; the slice reads as "the rule-based metric is conservative on short content."
- **`handoff_success_rate` is lowest for `tool_failure` (0.75) and `step_cap_hit` (0.78)** — by design. These categories include planned upstream failures; the handoff metric correctly reflects that downstream agents received `status="error"` envelopes.
- **`routing_accuracy = 0.967` aggregate** is dragged down by `happy_path` at 0.90 — the `lab10_t05` routing bug. The diagnostic surfaces immediately.

The discipline: aggregate first to find what's off, then slice to localize, then read individual traces to understand. Skipping the slice produces wrong conclusions.


## Step 8: Per-agent breakdown

The same trace data, re-keyed by which agent caused which signal. The supervisor is the routing-decision maker; the researcher produces findings; the writer composes prose; the critic gates (Lab 11); the planner emits plans (Lab 12).

Per-agent breakdown is most useful when an aggregate metric drops and you need to localize the cause to a specific agent. The end-to-end metric tells you the system has a problem; per-agent tells you which agent is dragging the metric.

In [ ]:
def per_agent_signals(trace_set: list[Trace]) -> pd.DataFrame:
    """Count failure signals attributable to each agent across the trace set."""
    rows = []
    for t in trace_set:
        for s in t.trajectory:
            row = {
                "trace_id": t.id,
                "node": s.node,
                "status": s.status,
                "is_error": s.status not in ("ok", "step_cap"),
                "is_step_cap": s.status == "step_cap",
            }
            rows.append(row)
    return pd.DataFrame(rows)


signal_df = per_agent_signals(traces)
print("Steps per agent type:")
print(signal_df["node"].value_counts())

print("\nError signals by agent:")
agent_signals = signal_df.groupby("node").agg(
    total_steps=("status", "size"),
    error_rate=("is_error", "mean"),
    step_cap_rate=("is_step_cap", "mean"),
).round(3).sort_values("error_rate", ascending=False)
print(agent_signals)


**Sample output**:

```
Steps per agent type:
node
supervisor     35
writer         13
executor       12
researcher     10
planner         9
critic          8
synthesizer     5

Error signals by agent:
             total_steps  error_rate  step_cap_rate
node
planner                9       0.444         0.111
researcher            10       0.100         0.100
executor              12       0.083         0.000
critic                 8       0.000         0.000
supervisor            35       0.000         0.000
synthesizer            5       0.000         0.000
writer                13       0.000         0.000
```

The breakdown tells you:

- **Planner**: 44% error rate. That's the validation-retry pattern — invalid plans get caught and replanned. High error_rate here is *expected* for a healthy validator; you'd worry if it were 0%.
- **Researcher**: 10% error rate (one rate-limit failure) and 10% step-cap rate (one ran out of budget). Both are real failure modes the harness should catch.
- **Executor**: 8% error rate (one tool failure on HTTP 503). Recoverable; the dispatcher routed around it.
- **Supervisor, writer, critic, synthesizer**: all 0% error rate. These agents handle their inputs in the trace fixtures without faulting. (In production, you'd expect non-zero rates as load grows; the trace set is designed to test specific failure modes, not all of them.)

The per-agent view is what makes iteration efficient: if the harness shows the *planner* dragging plan_validity down, you iterate on the planner prompt, not the executor. If the *researcher* is the slow agent, you don't waste time tuning the writer.


## Step 9 (optional): LLM-as-judge variant of plan validity

The rule-based `plan_validity` checks the `plan_valid` flag in the planner's output. That's deterministic but trusts the validator's logic. An LLM-as-judge variant reads the plan and the task and decides whether the plan is a *reasonable* approach — catching plans that are structurally valid but semantically off.

This step is optional. It costs ~$0.01 in API calls (one judge call per Lab 12 trace). Skip it if you don't want to spend the money or don't have credentials configured.

In [ ]:
def llm_judge_plan_validity(trace: Trace, llm_client=None) -> dict | None:
    """LLM-as-judge variant of plan_validity. Asks the LLM to score the
    plan's suitability for the task on 0/1.

    Returns None for non-Lab-12 traces or when no LLM client is provided.
    """
    if trace.source_lab != "lab12" or llm_client is None:
        return None

    planner_steps = [s for s in trace.trajectory if s.node == "planner"]
    if not planner_steps:
        return {"score": 0, "reasoning": "no planner step found"}

    final_plan = planner_steps[-1].output.get("plan", {})
    plan_repr = json.dumps(final_plan, indent=2)

    judge_prompt = f"""Evaluate whether this plan is suitable for the user's task.

USER TASK:
{trace.task}

PLAN (final attempt):
{plan_repr}

Score: 1 if the plan is a reasonable approach to the task, 0 if it isn't.
Respond ONLY with JSON: {{"score": 1, "reasoning": "<one sentence>"}}
"""
    # Provider-specific call
    from langchain_core.messages import SystemMessage, HumanMessage
    if PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        client = llm_client or ChatOpenAI(model=MODEL, temperature=0)
    else:
        from langchain_anthropic import ChatAnthropic
        client = llm_client or ChatAnthropic(model=MODEL, temperature=0)

    resp = client.invoke([
        SystemMessage(content="You are an evaluator. Respond only with the requested JSON."),
        HumanMessage(content=judge_prompt),
    ])
    raw = (resp.content or "").strip()
    # Strip code fences if present
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"score": -1, "reasoning": f"could not parse: {raw[:100]}"}


# Run on the two replan_needed traces (cheap demo)
print("LLM-as-judge plan validity vs rule-based:")
print(f"{'trace_id':<15} {'rule_based':>10} {'llm_judge':>10} {'judge_reasoning'}")
print("─" * 100)
for t in traces:
    if t.category != "replan_needed":
        continue
    rule = plan_validity(t)
    judge = llm_judge_plan_validity(t)
    if judge is None:
        continue
    reasoning = judge.get("reasoning", "?")[:60]
    print(f"{t.id:<15} {rule:>10.2f} {judge['score']:>10} {reasoning}")


**Sample output** (LLM responses will vary):

```
LLM-as-judge plan validity vs rule-based:
trace_id        rule_based  llm_judge  judge_reasoning
─────────────────────────────────────────────────────────────────────
lab12_t02            1.00          1  The plan correctly sequences search and fetches
lab12_t04            1.00          1  Plan retrieves an evaluation paper as requested
```

Both replan-needed traces score 1.0 on rule-based validity (because the final plan was structurally valid) and 1 from the LLM judge (because the final plan is semantically reasonable for the task). The agreement here is the typical case — when the plan validator says "valid" and the plan is well-formed, the LLM agrees.

The disagreement case (not in this trace set) is when a plan is structurally valid but doesn't actually solve the task. The rule-based version misses that; the LLM judge catches it. Trade-off: $0.005 per judge call, plus the Zheng et al. biases. For production, an LLM judge with periodic human calibration is the typical pattern.

## Step 10: Synthesis — what the harness revealed and what comes next

What this lab has built:

**A reusable evaluation harness** for multi-agent trajectories. Seven metrics from scratch (five trajectory, two outcome). Category-aware aggregation. Per-agent breakdown. Optional LLM-as-judge tier.

**A trace set as contract**. 15 hand-curated traces covering five categories. Modifying these to "fix" failing metrics is the wrong direction — the traces are the spec, the metrics are the implementation.

**A diagnostic discipline**. Aggregate metrics tell you "the system has a problem"; category slicing tells you "which kind of problem"; per-agent breakdown tells you "which agent to fix." All three layers are needed.

What the harness revealed in our trace set:

- The supervisor's routing is mostly correct (`routing_accuracy = 0.93`), but the `lab10_t05` routing bug (skipping the researcher) shows up clearly when sliced by trace.
- Citation preservation has a real failure mode in `citation_drift` traces (drops to 0.56). The aggregate `0.72` hid this; the slice surfaced it.
- The planner correctly catches invalid plans (`plan_validity = 1.0` on all completed Lab 12 traces) but exhausts retries on one ambiguous task (`lab12_t05`, `plan_validity = 0.0`).
- Per-agent breakdown localizes the work: the planner has the highest error rate (validator-rejection-and-retry pattern); the supervisor, writer, critic, synthesizer are clean.

What the harness *doesn't* tell you:

- **Whether the trace set represents production traffic.** The category distribution in `trace_set.jsonl` was hand-designed for diagnostic coverage. Production traces follow a different distribution (mostly happy_path; tool_failure rates vary by service reliability).
- **Whether the rule-based groundedness misses paraphrased-but-true claims.** It does. LLM-as-judge catches more of those at higher cost. Calibrate based on your tolerance for false-negative grounding signals.
- **Whether handoffs at scale degrade.** The replay model evaluates known traces; live degradation needs online evaluation.

What production needs that we didn't build:

- **Trace ingestion infrastructure.** LangSmith / Phoenix / OpenTelemetry-based. The harness here reads JSONL fixtures; production reads spans from live agents.
- **Online evaluation and drift detection.** Score traces as they come in; alert when category-level metrics drop.
- **Agent-as-judge calibration.** The Zheng et al. (2023) biases — position, verbosity, self-enhancement — need periodic calibration against human ground truth.
- **Multi-turn evaluation.** Conversational agents need trajectory metrics across conversation turns. LangSmith's multi-turn evals (Oct 2025) is the documented path.

The harness here is the conceptual foundation. Production tooling layers on top of it — but the metric implementations and the diagnostic discipline (slice by category, breakdown by agent) carry over directly.

✓ **Path 03 v1 complete.** Foundations → patterns (supervisor-worker, generator-critic, plan-and-execute, multi-agent RAG) → framework bridge → evaluation. The path is structurally closed; further extensions (Path 06 production observability, Path 03 v2 multi-turn, framework-bridge solutions) build on top.
